In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
import time as tm
from time import time
import fitsio
import pandas as pd
from galpy.potential import MWPotential2014, vcirc, evaluateRforces
from galpy.orbit import Orbit
from scipy.stats import gaussian_kde, ks_2samp, kstest
from hyppo.ksample import Energy
import corner
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

from astroquery.gaia import Gaia
from astropy.table import Table
from astropy.table import Table, vstack
from glob import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LinearRegression
import os
from datetime import datetime
from matplotlib.ticker import AutoLocator
import re

Maintenance with possible short-time disconnections: 29 June 2026 18:00–20:00 CEST


In [2]:
# Setup and helper functions moved to external module
from plots_for_ASA_cell2 import *

# If you change the external module and want live reload during development, uncomment:
# %load_ext autoreload
# %autoreload 2

In [3]:
Vmans_NGC3201 = pd.read_csv(
    Vmans_NGC3201_txt,
    sep='\t',
    header=0
)
Vmans_NGC3201.columns = Vmans_NGC3201.columns.str.replace('#', '').str.strip()
Vmans_NGC3201_high_prob = Vmans_NGC3201[(Vmans_NGC3201['memberprob'] > 0.99)]# & (Vmans_NGC3201['qflag'] == 0)]



Vmans_NGC5139 = pd.read_csv(
    Vmans_NGC5139_txt,
    sep='\t',
    header=0
)
Vmans_NGC5139.columns = Vmans_NGC5139.columns.str.replace('#', '').str.strip()
Vmans_NGC5139_high_prob = Vmans_NGC5139[(Vmans_NGC5139['memberprob'] > 0.99)]# & (Vmans_NGC5139['qflag'] == 0)]


Vmans_NGC1851 = pd.read_csv(
    Vmans_NGC1851_txt,
    sep='\t',
    header=0
)
Vmans_NGC1851.columns = Vmans_NGC1851.columns.str.replace('#', '').str.strip()
Vmans_NGC1851_high_prob = Vmans_NGC1851[(Vmans_NGC1851['memberprob'] > 0.99)]# & (Vmans_NGC5139['qflag'] == 0)]

In [4]:
with fitsio.FITS(galah_Gaia_fits) as hdul1:
    data_1 = hdul1[1].read()
galah_Gaia_raw = pd.DataFrame(data_1)
galah_Gaia_raw = ensure_native_endian(galah_Gaia_raw)

In [5]:
def NGC3201_cuts_pradosh(df):  # I would need to make some small changes for this to work for galah. really Ijust need to change the RA DEC and Energy to the correct column name
      return((152 < df['RA']) & (156 > df['RA']) & (-48 < df['DEC']) & (-45 > df['DEC']) & (2000 > df['Energy']) & (-2500 > df['jphi']))
     
def Sequoia_cuts_Diane_2021(df):
    return (-1.0< df['jphi']/df['jtot']) & (df['jphi']/df['jtot']<-0.6) & (-1.0 < (df['jz'] - df['jr'])/df['jtot']) & (-1.0 < (df['jz'] - df['jr'])/df['jtot']) & (0.1 > (df['jz'] - df['jr'])/df['jtot'])

def GSE_cuts_Diane_2021(df):
        return (-500< df['jphi']) & (500 > df['jphi']) & (30 < np.sqrt(df['jr'])) & (55 > np.sqrt(df['jr']))

In [6]:
galah_Gaia = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.2)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['flag_mg_fe'] == 0)
                        # & (galah_Gaia_raw['flag_na_fe'] == 0)
                        # & (galah_Gaia_raw['flag_cu_fe'] == 0)
                        & (galah_Gaia_raw['fe_h'] < -0.75)
                        # & (galah_Gaia_raw['fe_h'] < -0.9)
                        # & (galah_Gaia_raw['ti_fe'] > 0.25)
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
    ]

In [ ]:
NGC3201_galah = pd.merge( galah_Gaia, Vmans_NGC3201_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC5139_galah = pd.merge( galah_Gaia, Vmans_NGC5139_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC1851_galah = pd.merge( galah_Gaia, Vmans_NGC1851_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')

galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)
                                 & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                 & (galah_Gaia_raw['flag_mg_fe'] == 0)
                                 & (galah_Gaia_raw['flag_na_fe'] == 0)
                                 & (galah_Gaia_raw['flag_cu_fe'] == 0)
                                 & ((galah_Gaia_raw['mg_fe'] - galah_Gaia_raw['cu_fe']) - ((1.06923) * galah_Gaia_raw['na_fe']) > 0.40150)
                                 ]

# Kushniruk et al . 2026 https://doi.org/10.1051/0004-6361/202451201
galah_Kushniruk_Thamnos1 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])- 13.5)**2) / (4.0**2)) + (((galah_Gaia['jphi'] + 600)**2) / (350**2)) < 1)
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

galah_Kushniruk_Thamnos2 = galah_Gaia[(((np.sqrt(galah_Gaia['jr'])-8.6)**2) / (3.5**2)) + (((galah_Gaia['jphi'] + 1184)**2) / (175**2)) < 1
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_GSE = galah_Gaia[GSE_cuts_Diane_2021(galah_Gaia)
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            # & (galah_Gaia['fe_h'] < -0.8)
                            ]

galah_Gaia_Sequoia = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            # & (galah_Gaia['fe_h'] < -0.8)
                            ]

galah_Gaia_Halo = galah_Gaia[(np.sqrt((galah_Gaia['vphi']-230)**2 + (galah_Gaia['vr'])**2 + (galah_Gaia['vz'])**2) > 230) # Koppelman et al. 2019  https://doi.org/10.1051/0004-6361/201936738
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                            # & (galah_Gaia_raw['ti_fe'] > 0.25)
                             ]

/tmp/ipykernel_26724/174340636.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)



In [8]:
element_list_full_galah = ["h", "fe", "n", "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti" "v", "cr" ,"mn", "co", "ni", "cu", "zn", "rb", "sr", "y", "zr", "mo", "ru", "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.
element_list_partial_small_galah_1 = [ 'fe', 'ca', 'ti', 'na', 'mn', 'cu','mg']
element_list_partial_small_galah_2 = [ 'fe', 'ca', 'na', 'mn', 'cu','mg', 'nd']
element_list_partial_small_galah_3 = [ 'cu', 'ca', 'mn', 'ni','v','na', 'nd']
element_list_partial_small_galah_4 = [ 'cu', 'ca', 'mn', 'ti','v','na', 'nd']
element_list_partial_small_galah_5 = [ 'cu', 'ca', 'mn', 'ti','ba','na', 'nd']
element_list_partial_small_galah_6 = [ 'cu', 'ca', 'mn', 'ti', 'v', 'ba','na', 'nd']
element_list_partial_small_galah_7 = [ 'fe', 'cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_2 = ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_3 = ['fe', 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_4 = ['fe', 'si', 'nd', 'na', 'mg', 'k', 'ca', 'sc', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba']
element_list_partial_small_galah_7_5 = ['v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_6 = ['v', 'k', 'na', 'mg', 'si', 'ca', 'sc', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_8 = [ 'ca', 'mn', 'y', 'v', 'na', 'nd', 'cr', 'sc']
element_list_partial_small_galah_9 = [ 'cu', 'ca', 'mn', 'ti', 'sc', 'v', 'na', 'nd']
element_list_partial_small_galah_10 = ['fe', 'v', 'sc', 'si', 'ca', 'mn', 'na', 'nd']
element_list_partial_small_galah_12 = ['ca', 'sc', 'na', 'mg', 'si', 'k', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_11 = ['y', 'ba', 'nd', 'eu']


elements_in_Pradosh_Galah_1 = ['ca', 'ti', 'ni','zr', 'ce', 'nd']
elements_in_Pradosh_Galah_2 = ['ca', 'ti', 'ni', 'nd']

element_list = element_list_partial_small_galah_7_5

# element_list_X_frequ = list(X_frequ['element'])
# element_list = element_list_X_frequ

galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia, element_list)

galah_Gaia_new_ratios = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0] # this ontains a full catalog from galah but now it includes ratios for Ca, Si, Ti, Ni, Zr, Ce, and Nd relative to eachother: Pradosh did recomend taking out
                                                                 # Si, and Ni whitch I can do by changing "element_list_partial_large" to element_list_partial_small
galah_gaia_new_element_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]     # this just lists the new columns ratios so I can more easily call them
galah_gaia_new_element_ratio_errors_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[2] 

galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_GSE, element_list)
galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_GSE, element_list)
galah_Gaia_Kushniruk_GSE_new_ratios = galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia, element_list)
galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC3201_galah, element_list)
# galah_Gaia_NGC3201_new_ratios = galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC5139_galah, element_list)
# galah_Gaia_NGC5139_new_ratios = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos1, element_list)
# galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos2, element_list)
# galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC1851_galah, element_list)
# galah_Gaia_NGC1851_new_ratios = galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Halo, element_list)
galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/Jupyter Notebooks/plots_for_ASA_cell2.py:556: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_set_flagged[new_ratio_error_label] = np.sqrt((data_set_flagged[err_col_numerator])**2 + (data_set_flagged[err_col_denominator])**2)

/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/Jupyter Notebooks/plots_for_ASA_cell2.py:555: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_set_flagged[new_ratio_label] = data_set_flagged[col_name_for_numerator] - data_set_flag

In [9]:
print(len(galah_Gaia_Sequoia_new_ratios))
print(len(galah_Gaia_GSE_new_ratios))
print(len(galah_Gaia_Halo_new_ratios))

65
527
2026


In [23]:
ratio_list = galah_gaia_new_element_ratio_list
df1 = galah_Gaia_Sequoia_new_ratios
# df2 = galah_Gaia_GSE_new_ratios
df2 = galah_Gaia_Halo_new_ratios

In [24]:
final_filtered_elm_ratio_list = find_ratio_combination_to_try_for_2d_KS_test(ratio_list, df1, df2, element_list, percent_value = 0.3, keep_num_elm = 4, do_costom_elm_filter = False, costom_elm_filter = ['si', 'sc', 'v', 'co', 'nd'])

percent_value =  0.3
len(ratio_1d_ks_test)  136
len(top_percent)  40
v and ca_v
v and sc_v
v and ti_v
v and k_v
v and mg_v
v and si_v
v and zn_v
v and cr_v
v and ni_v
v and mn_v
v and y_v
v and cu_v
v and co_v
v and nd_v
v and na_v
v and ba_v
sc and sc_v
sc and na_sc
sc and cu_sc
sc and co_sc
sc and ni_sc
sc and ti_sc
sc and ba_sc
sc and cr_sc
sc and mn_sc
sc and nd_sc
sc and mg_sc
sc and si_sc
sc and y_sc
sc and ca_sc
sc and k_sc
sc and zn_sc
na and na_sc
na and k_na
na and zn_na
na and si_na
na and ca_na
na and ti_na
na and mg_na
na and nd_na
na and cr_na
na and y_na
na and ni_na
na and mn_na
na and ba_na
na and co_na
na and cu_na
na and na_v
mg and mg_v
mg and co_mg
mg and ni_mg
mg and cu_mg
mg and mn_mg
mg and mg_sc
mg and k_mg
mg and ba_mg
mg and mg_na
mg and ca_mg
mg and nd_mg
mg and zn_mg
mg and cr_mg
mg and y_mg
mg and ti_mg
mg and si_mg
si and si_v
si and cu_si
si and si_na
si and ni_si
si and co_si
si and k_si
si and mn_si
si and si_sc
si and ba_si
si and nd_si
si and ca_si
s

,element,frequency,score
5,k,8,-107.182085
0,v,7,-106.703021
6,ca,7,-101.478570
1,sc,9,-93.360770
12,cu,7,-79.439001
11,ni,6,-66.104644
10,co,5,-65.575120
7,ti,4,-63.757456
2,na,4,-62.626729
13,zn,5,-60.367189


auto_elm_filter =  ['k', 'v', 'ca', 'sc']
28


,ratio,p_value,statistic
5,ca_v,5.123140e-10,0.408748
0,sc_v,4.549937e-06,0.315461
6,ti_v,5.669002e-06,0.312826
16,na_sc,7.268630e-06,0.309788
33,k_na,7.788598e-06,0.308960
84,co_ca,8.129484e-06,0.308452
75,ni_k,1.389169e-05,0.301800
86,cu_ca,1.677996e-05,0.299415
72,cr_k,1.937246e-05,0.297600
4,k_v,2.870104e-05,0.292513


In [25]:
print(len(final_filtered_elm_ratio_list))

final_filtered_elm_ratio_list

28


['ca_v',
 'sc_v',
 'ti_v',
 'na_sc',
 'k_na',
 'co_ca',
 'ni_k',
 'cu_ca',
 'cr_k',
 'k_v',
 'cu_sc',
 'co_k',
 'cu_k',
 'mg_v',
 'si_v',
 'co_sc',
 'ni_sc',
 'mn_k',
 'mn_ca',
 'zn_v',
 'cr_ca',
 'ba_k',
 'ni_ca',
 'ti_sc',
 'ba_sc',
 'cr_sc',
 'ba_ca',
 'mn_sc']

In [26]:
two_by_two_combinations = make_2_vs_2_combinations(final_filtered_elm_ratio_list, df1, df2, element_list)  #this returns the set of 2 by 2 combinations, updated df1, and updated df2


len of combined_ratios_list is 378
['ca_v_pl_sc_v', 'ca_v_pl_ti_v', 'ca_v_min_na_sc', 'ca_v_pl_k_na', 'ca_v_min_co_ca', 'ca_v_min_ni_k', 'ca_v_min_cu_ca', 'ca_v_min_cr_k', 'ca_v_pl_k_v', 'ca_v_min_cu_sc', 'ca_v_min_co_k', 'ca_v_min_cu_k', 'ca_v_pl_mg_v', 'ca_v_pl_si_v', 'ca_v_min_co_sc', 'ca_v_min_ni_sc', 'ca_v_min_mn_k', 'ca_v_min_mn_ca', 'ca_v_pl_zn_v', 'ca_v_min_cr_ca', 'ca_v_min_ba_k', 'ca_v_min_ni_ca', 'ca_v_min_ti_sc', 'ca_v_min_ba_sc', 'ca_v_min_cr_sc', 'ca_v_min_ba_ca', 'ca_v_min_mn_sc', 'sc_v_pl_ti_v', 'sc_v_min_na_sc', 'sc_v_pl_k_na', 'sc_v_min_co_ca', 'sc_v_min_ni_k', 'sc_v_min_cu_ca', 'sc_v_min_cr_k', 'sc_v_pl_k_v', 'sc_v_min_cu_sc', 'sc_v_min_co_k', 'sc_v_min_cu_k', 'sc_v_pl_mg_v', 'sc_v_pl_si_v', 'sc_v_min_co_sc', 'sc_v_min_ni_sc', 'sc_v_min_mn_k', 'sc_v_min_mn_ca', 'sc_v_pl_zn_v', 'sc_v_min_cr_ca', 'sc_v_min_ba_k', 'sc_v_min_ni_ca', 'sc_v_min_ti_sc', 'sc_v_min_ba_sc', 'sc_v_min_cr_sc', 'sc_v_min_ba_ca', 'sc_v_min_mn_sc', 'ti_v_min_na_sc', 'ti_v_pl_k_na', 'ti_v_min_co_ca'

In [27]:
two_by_two_combinations[0]

[['ca_v_pl_sc_v', 'k_na_min_ni_k'],
 ['ca_v_pl_sc_v', 'k_na_min_cr_k'],
 ['ca_v_pl_sc_v', 'k_na_min_co_k'],
 ['ca_v_pl_sc_v', 'k_na_min_cu_k'],
 ['ca_v_pl_sc_v', 'k_na_min_mn_k'],
 ['ca_v_pl_sc_v', 'k_na_min_ba_k'],
 ['ca_v_pl_sc_v', 'ni_k_pl_cr_k'],
 ['ca_v_pl_sc_v', 'ni_k_pl_co_k'],
 ['ca_v_pl_sc_v', 'ni_k_pl_cu_k'],
 ['ca_v_pl_sc_v', 'ni_k_pl_mn_k'],
 ['ca_v_pl_sc_v', 'ni_k_pl_ba_k'],
 ['ca_v_pl_sc_v', 'cr_k_pl_co_k'],
 ['ca_v_pl_sc_v', 'cr_k_pl_cu_k'],
 ['ca_v_pl_sc_v', 'cr_k_pl_mn_k'],
 ['ca_v_pl_sc_v', 'cr_k_pl_ba_k'],
 ['ca_v_pl_sc_v', 'co_k_pl_cu_k'],
 ['ca_v_pl_sc_v', 'co_k_pl_mn_k'],
 ['ca_v_pl_sc_v', 'co_k_pl_ba_k'],
 ['ca_v_pl_sc_v', 'cu_k_pl_mn_k'],
 ['ca_v_pl_sc_v', 'cu_k_pl_ba_k'],
 ['ca_v_pl_sc_v', 'mn_k_pl_ba_k'],
 ['ca_v_pl_ti_v', 'na_sc_min_k_na'],
 ['ca_v_pl_ti_v', 'na_sc_pl_ni_k'],
 ['ca_v_pl_ti_v', 'na_sc_pl_cr_k'],
 ['ca_v_pl_ti_v', 'na_sc_pl_cu_sc'],
 ['ca_v_pl_ti_v', 'na_sc_pl_co_k'],
 ['ca_v_pl_ti_v', 'na_sc_pl_cu_k'],
 ['ca_v_pl_ti_v', 'na_sc_pl_co_sc'],
 ['c

In [28]:
print(len(two_by_two_combinations[0]))

12263


In [29]:
print((two_by_two_combinations[0][0:60]))

[['ca_v_pl_sc_v', 'k_na_min_ni_k'], ['ca_v_pl_sc_v', 'k_na_min_cr_k'], ['ca_v_pl_sc_v', 'k_na_min_co_k'], ['ca_v_pl_sc_v', 'k_na_min_cu_k'], ['ca_v_pl_sc_v', 'k_na_min_mn_k'], ['ca_v_pl_sc_v', 'k_na_min_ba_k'], ['ca_v_pl_sc_v', 'ni_k_pl_cr_k'], ['ca_v_pl_sc_v', 'ni_k_pl_co_k'], ['ca_v_pl_sc_v', 'ni_k_pl_cu_k'], ['ca_v_pl_sc_v', 'ni_k_pl_mn_k'], ['ca_v_pl_sc_v', 'ni_k_pl_ba_k'], ['ca_v_pl_sc_v', 'cr_k_pl_co_k'], ['ca_v_pl_sc_v', 'cr_k_pl_cu_k'], ['ca_v_pl_sc_v', 'cr_k_pl_mn_k'], ['ca_v_pl_sc_v', 'cr_k_pl_ba_k'], ['ca_v_pl_sc_v', 'co_k_pl_cu_k'], ['ca_v_pl_sc_v', 'co_k_pl_mn_k'], ['ca_v_pl_sc_v', 'co_k_pl_ba_k'], ['ca_v_pl_sc_v', 'cu_k_pl_mn_k'], ['ca_v_pl_sc_v', 'cu_k_pl_ba_k'], ['ca_v_pl_sc_v', 'mn_k_pl_ba_k'], ['ca_v_pl_ti_v', 'na_sc_min_k_na'], ['ca_v_pl_ti_v', 'na_sc_pl_ni_k'], ['ca_v_pl_ti_v', 'na_sc_pl_cr_k'], ['ca_v_pl_ti_v', 'na_sc_pl_cu_sc'], ['ca_v_pl_ti_v', 'na_sc_pl_co_k'], ['ca_v_pl_ti_v', 'na_sc_pl_cu_k'], ['ca_v_pl_ti_v', 'na_sc_pl_co_sc'], ['ca_v_pl_ti_v', 'na_sc_pl_ni_s

In [30]:
two_by_two_combinations[2]#[two_by_two_combinations[0][1]]

,sobject_id,tmass_id,gaiadr3_source_id,survey_name,field_id,setup,mjd,ra,dec,flag_sp,...,ti_sc_pl_ba_sc,ti_sc_pl_cr_sc,ti_sc_pl_ba_ca,ti_sc_pl_mn_sc,ba_sc_pl_cr_sc,ba_sc_pl_ba_ca,ba_sc_pl_mn_sc,cr_sc_pl_ba_ca,cr_sc_pl_mn_sc,ba_ca_pl_mn_sc
5769,151009004101166,03254556+1810597,55930202296881280,k2_hermes,6602,allstar,57304.700,51.439835,18.183250,0,...,0.125361,-0.083150,0.006918,-0.344703,-0.276010,-0.185941,-0.537562,-0.394453,-0.746074,-0.656006
10731,150408002901065,08301838+1026373,600068817434968960,k2_hermes,6614,allstar,57120.395,127.576584,10.443722,0,...,0.269828,-0.136463,0.156103,-0.301365,-0.093599,0.198967,-0.258501,-0.207323,-0.664792,-0.372226
13598,160112002301024,08424236+1342390,609143911534194944,k2_hermes,6610,allstar,57399.625,130.676544,13.710861,0,...,0.027314,-0.087547,-0.113808,-0.257449,-0.224378,-0.250639,-0.394280,-0.365499,-0.509141,-0.535402
15667,160109003301383,08301587+1408400,651626184677599744,k2_hermes,6615,allstar,57396.640,127.566124,14.144417,0,...,0.307051,0.081662,0.136626,-0.107261,-0.058417,-0.003452,-0.247340,-0.228841,-0.472729,-0.417764
16126,160331001701267,08265545+1445438,652469926709578624,k2_hermes,6615,allstar,57478.406,126.731087,14.762167,0,...,0.045276,-0.131911,-0.095751,-0.287272,-0.225125,-0.188966,-0.380486,-0.366152,-0.557673,-0.521513
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916286,210524004201173,20512007-1103149,6901934722944726400,galah_main,654,allstar,59358.785,312.833618,-11.054194,0,...,0.340186,-0.028394,0.128174,-0.233442,-0.086684,0.069885,-0.291732,-0.298696,-0.660312,-0.503744
916488,210524004201290,20531743-1001558,6902430057228646656,galah_main,654,allstar,59358.785,313.322723,-10.032166,0,...,0.156741,0.051551,-0.045903,-0.103220,-0.170962,-0.268417,-0.325734,-0.373607,-0.430924,-0.528378
916669,150601004801246,20453056-0913097,6903163495907475328,galah_main,825,allstar,57174.785,311.377319,-9.219389,0,...,0.675998,0.012110,0.381168,-0.181250,0.240099,0.609156,0.046738,-0.054732,-0.617150,-0.248092
916713,150601004801392,20491708-0925257,6903458478557543168,galah_main,825,allstar,57174.785,312.321167,-9.423833,0,...,0.342127,0.073794,0.119277,-0.264924,-0.025013,0.020471,-0.363731,-0.247862,-0.632063,-0.586580


In [31]:

def make_2_vs_2_2d_ks_tests(combination_val):
    ratio_ks_test = pd.DataFrame(columns=['ratio_com_1', 'ratio_com_2', 'p_value', 'statistic'])

    df1 = combination_val[1]
    df2 = combination_val[2]

    # for i in range(0,100):
    for i in range(len(combination_val[0])):
        ratio_com_1 = combination_val[0][i][0]
        # print('ratio_com_1 ', ratio_com_1)
        ratio_com_2 = combination_val[0][i][1]
        # print('ratio_com_2 ', ratio_com_2)
        print(f'{ratio_com_1} and {ratio_com_2}')
                # example datasets
            # keep paired finite values
    
        sample1 = df1[[ratio_com_1, ratio_com_2]].dropna()
        sample2 = df2[[ratio_com_1, ratio_com_2]].dropna()
        
        # convert to numpy arrays
        X = sample1.to_numpy()
        Y = sample2.to_numpy()
        # multivariate comparison
        statistic, p_value = Energy().test(X, Y)

        ratio_ks_test.loc[len(ratio_ks_test)] = [
            ratio_com_1,
            ratio_com_2,
            p_value,
            statistic
        ]
    ratio_ks_test = ratio_ks_test.sort_values(
        by='p_value',
        ascending=True
        )
    return ratio_ks_test


In [32]:
two_by_two_table = make_2_vs_2_2d_ks_tests(two_by_two_combinations)

ca_v_pl_sc_v and k_na_min_ni_k
ca_v_pl_sc_v and k_na_min_cr_k
ca_v_pl_sc_v and k_na_min_co_k
ca_v_pl_sc_v and k_na_min_cu_k
ca_v_pl_sc_v and k_na_min_mn_k
ca_v_pl_sc_v and k_na_min_ba_k
ca_v_pl_sc_v and ni_k_pl_cr_k
ca_v_pl_sc_v and ni_k_pl_co_k
ca_v_pl_sc_v and ni_k_pl_cu_k
ca_v_pl_sc_v and ni_k_pl_mn_k
ca_v_pl_sc_v and ni_k_pl_ba_k
ca_v_pl_sc_v and cr_k_pl_co_k
ca_v_pl_sc_v and cr_k_pl_cu_k
ca_v_pl_sc_v and cr_k_pl_mn_k
ca_v_pl_sc_v and cr_k_pl_ba_k
ca_v_pl_sc_v and co_k_pl_cu_k
ca_v_pl_sc_v and co_k_pl_mn_k
ca_v_pl_sc_v and co_k_pl_ba_k
ca_v_pl_sc_v and cu_k_pl_mn_k
ca_v_pl_sc_v and cu_k_pl_ba_k
ca_v_pl_sc_v and mn_k_pl_ba_k
ca_v_pl_ti_v and na_sc_min_k_na
ca_v_pl_ti_v and na_sc_pl_ni_k
ca_v_pl_ti_v and na_sc_pl_cr_k
ca_v_pl_ti_v and na_sc_pl_cu_sc
ca_v_pl_ti_v and na_sc_pl_co_k
ca_v_pl_ti_v and na_sc_pl_cu_k
ca_v_pl_ti_v and na_sc_pl_co_sc
ca_v_pl_ti_v and na_sc_pl_ni_sc
ca_v_pl_ti_v and na_sc_pl_mn_k
ca_v_pl_ti_v and na_sc_pl_ba_k
ca_v_pl_ti_v and na_sc_pl_ba_sc
ca_v_pl_ti_v and n

In [33]:
two_by_two_table

,ratio_com_1,ratio_com_2,p_value,statistic
4509,na_sc_min_si_v,co_k_pl_cr_ca,1.789438e-10,0.088693
4470,na_sc_min_si_v,co_ca_pl_cr_k,1.789438e-10,0.088693
6772,co_ca_pl_cr_k,si_v_min_ni_sc,2.869587e-10,0.076532
10737,co_k_pl_cr_ca,si_v_min_ni_sc,2.869587e-10,0.076532
4473,na_sc_min_si_v,co_ca_pl_mn_k,2.947684e-10,0.091390
...,...,...,...,...
3843,ti_v_min_ti_sc,ba_k_pl_ba_ca,1.393330e-04,0.052941
12010,co_sc_pl_ti_sc,ba_k_pl_ba_ca,1.450903e-04,0.051774
12085,ni_sc_pl_ti_sc,ba_k_pl_ba_ca,1.685016e-04,0.050841
12258,ba_k_pl_ba_ca,ti_sc_pl_mn_sc,1.968618e-04,0.049431


In [34]:
two_by_two_table.to_csv('two_by_two_ks_test_no_X_vs_X_phase_1_2_Hermes_fe_lessthen_min0.75_H_vs_S_.csv', index=False)